Example Test Query Results

In [0]:
from pyspark.sql import functions as F
catalog = "rd_edapt"
schema = "analytics_bronze"
table = "pfd_obligations"
full_table_name = f"{catalog}.{schema}.{table}"

pfdo = spark.table(full_table_name)
pfdo.show(10)

[copied from readme]
People generally request RD information for a geographical area or type of business. So, our examples consider this, and the tools were specifically designed to answer these kinds of questions.

Test Examples:

1. What was the total investment in Washington state in FY 2024? [Total $ in FY for geography]
2. What portion of the investment in Washington state in FY 2024 was for water related projects? [subset]
3. Which North Carolina county received the most RD investment in FY 2024? [Specific 'top'/ relative geography]
4. From FY 2021 through FY 2025, how much did RD invest in Illinois? [aggregate across years]
5. Which kinds of businesses received the most RD investment for Rural Business in FY 2026? [sector naics]
6. South Dakota had severe storms from June 28 - July 3, 2026. What portion of the South Dakota housing investment from RD (both direct and held by other lenders) was impacted? A user will probably not know the disaster declaration number, but for reference, this is South Dakota Severe Storms, Straight-line Winds, Tornadoes, and Flooding.

## 1. What was the total investment in Washington state in FY 2024? [Total $ in FY for geography]

In [0]:
q1 = pfdo.filter(pfdo.obligation_state_abr == 'WA')\
         .filter(pfdo.obl_fscl_yr == '2024')\
         .groupBy('state_name','obl_fscl_yr')\
         .agg(F.sum('obl_amt').alias('total_investment'))\


display(q1)


## 2. What portion of the investment in Washington state in FY 2024 was for water related projects? [subset]

In [0]:
q2 = pfdo.filter(pfdo.obligation_state_abr == 'WA')\
         .filter(pfdo.obl_fscl_yr == '2024')\
         .groupBy('state_name','obl_fscl_yr', 'programarea')\
         .agg(F.sum('obl_amt').alias('total_investment'))
q2_tot = q2.select(F.sum('total_investment')).collect()[0][0]
q2 = q2.withColumn('portion_of_total', F.round(F.col('total_investment') / q2_tot,2))

q2_water = q2.filter(q2.programarea == 'Water and Environmental')

display(q2_water)

## 3. Which North Carolina county received the most RD investment in FY 2024? [Specific 'top'/ relative geography]

This is an interesting case because naively, we would want to get the top county, but the top county could be 'Not Available' because many large infrastructure investments cross counties. When we build our MCP tool, we will have to make sure that if the top returned is 'Not Available' that it gives more context to the user and provides the second result as well. 

In [0]:
q3 = pfdo.filter(pfdo.obligation_state_abr == 'NC')\
         .filter(pfdo.obl_fscl_yr == '2024')\
         .groupBy('state_name','obl_fscl_yr', 'obligation_county_name')\
         .agg(F.sum('obl_amt').alias('total_investment'))\
         .orderBy(F.col('total_investment').desc())
q3_tot = q3.select(F.sum('total_investment')).collect()[0][0]
q3 = q3.withColumn('portion_of_total', F.round(F.col('total_investment') / q3_tot,2))

q3_top_naive = q3.limit(1).collect()[0]
if q3_top_naive['obligation_county_name'] == 'Not Available':
    q3_top_county = q3.filter(q3.obligation_county_name != 'Not Available').limit(1).collect()[0]
else: 
    q3_top_county = q3_top_naive

display(q3_top_naive)

display(q3_top_county)

## 4. From FY 2021 through FY 2025, how much did RD invest in Illinois? [aggregate across years]

In [0]:
q4 = pfdo.filter(pfdo.obligation_state_abr == 'IL')\
         .withColumn("fy", F.col("obl_fscl_yr").cast("int"))\
         .filter(F.col("fy").between(2021, 2025))\
         .groupBy('state_name','obl_fscl_yr')\
         .agg(F.sum('obl_amt').alias('total_investment'))\
         .orderBy(F.col('obl_fscl_yr'))\
         .withColumn("obl_fscl_yr", F.col("obl_fscl_yr").cast("string"))
q4_total = q4.groupBy('state_name')\
             .agg(F.sum('total_investment').alias('total_investment'))\
             .withColumn("obl_fscl_yr", F.lit("Total"))

q4 = q4.unionByName(q4_total)

display(q4)

## 5. Which kinds of businesses received the most RD investment for Rural Business in FY 2026? [sector naics]

In [0]:

# get the business investment for fy26 with sector detail
q5 = pfdo.filter(pfdo.portfolio_group == 'Commercial')\
         .filter(pfdo.obl_fscl_yr == '2026')\
         .groupBy('obl_fscl_yr', 'naics_industry_sector')\
         .agg(F.sum('obl_amt').alias('total_investment'))\
         .orderBy(F.col('total_investment').desc())
q5_tot = q5.select(F.sum('total_investment')).collect()[0][0]
q5 = q5.withColumn('portion_of_total', F.round(F.col('total_investment') / q5_tot,2))

# get the top business investment sector with caution about the 'other' sector
q5_top_naive = q5.limit(1).collect()[0]
if q5_top_naive['naics_industry_sector'] == 'Other or Not Defined':
    q5_top = q5.filter(q5.naics_industry_sector != 'Other or Not Defined').limit(1).collect()[0]
else: 
    q5_top = q5_top_naive

display(q5_top_naive)

display(q5_top)

## 6. South Dakota had severe storms from June 28 - July 3, 2026. What portion of the South Dakota housing investment from RD (both direct and held by other lenders) was impacted? A user will probably not know the disaster declaration number, but for reference, this is South Dakota Severe Storms, Straight-line Winds, Tornadoes, and Flooding.

In [0]:
# we need to get the disaste rstorm counties
import requests # normally we do this at top of file but wtvr
import sys
FEMA_API_URL = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"

# need to be able to get a disaster number based on a description thing cuz users will not know this
# they prob know year tho
import requests
import pandas as pd
from datetime import date


FEMA_URL = (
    "https://www.fema.gov/api/open/v2/"
    "DisasterDeclarationsSummaries"
)


def get_fema_disasters(
    state: str,
    title_contains: str,
    incident_begin_date: str | date,
    incident_end_date: str | date,
    timeout: int = 30,
) -> pd.DataFrame:
    """
    Query FEMA disaster declarations and return selected fields as a DataFrame.

    Parameters
    ----------
    state : str
        Two-letter state abbreviation, such as "SD".
    title_contains : str
        Text to search for in declarationTitle, such as "STORM".
    incident_begin_date : str or datetime.date
        Earliest incident begin date, formatted as YYYY-MM-DD.
    incident_end_date : str or datetime.date
        Latest incident end date, formatted as YYYY-MM-DD.
    timeout : int
        HTTP request timeout in seconds.

    Returns
    -------
    pandas.DataFrame
        Columns:
        disasterNumber, declarationTitle,
        incidentBeginDate, incidentEndDate
    """

    def format_date(value):
        if isinstance(value, date):
            return value.isoformat()
        return str(value)

    state = state.upper().strip()
    title_contains = title_contains.upper().strip()
    begin_date = format_date(incident_begin_date)
    end_date = format_date(incident_end_date)

    # FEMA's API uses OData filter syntax.
    odata_filter = (
        f"state eq '{state}' "
        f"and substringof('{title_contains}',declarationTitle) "
        f"and incidentBeginDate ge {begin_date} "
        f"and incidentEndDate le {end_date}"
    )

    params = {
        "$filter": odata_filter,
        "$select": (
            "disasterNumber,declarationTitle,"
            "incidentBeginDate,incidentEndDate"
        ),
        "$orderby": "incidentBeginDate asc",
        "$top": 1000,
    }

    response = requests.get(FEMA_URL, params=params, timeout=timeout)
    response.raise_for_status()

    records = response.json().get("DisasterDeclarationsSummaries", [])

    columns = [
        "disasterNumber",
        "declarationTitle",
        "incidentBeginDate",
        "incidentEndDate",
    ]

    return pd.DataFrame(records, columns=columns).drop_duplicates(subset="disasterNumber").reset_index(drop=True)

# function to get counties (might need to do something diff idk)
def get_disaster_areas(
    disaster_number: int | str,
    timeout: int = 30,
) -> pd.DataFrame:
    """
    Return FEMA designated areas for one disaster declaration.

    Parameters
    ----------
    disaster_number : int or str
        FEMA disaster number, such as 1234.
    timeout : int
        HTTP request timeout in seconds.

    Returns
    -------
    pandas.DataFrame
        One row per FEMA designated area.
    """

    disaster_number = int(disaster_number)

    filters = [f"disasterNumber eq {disaster_number}"]

    params = {
        "$filter": " and ".join(filters),
        "$select": (
            "disasterNumber,state,designatedArea,"
            "fipsStateCode,fipsCountyCode,placeCode,"
            "declarationType"
        ),
        "$orderby": "state asc,designatedArea asc",
        "$top": 1000,
    }

    response = requests.get(
        FEMA_URL,
        params=params,
        timeout=timeout,
    )
    response.raise_for_status()

    records = response.json().get(
        "DisasterDeclarationsSummaries",
        [],
    )

    columns = [
        "disasterNumber",
        "state",
        "designatedArea",
        "fipsStateCode",
        "fipsCountyCode",
        "placeCode"
    ]

    areas = pd.DataFrame(records, columns=columns)

    # Remove duplicate rows, if any.
    return areas.drop_duplicates().reset_index(drop=True)

In [0]:
# test the disaster number
oohtmp = get_fema_disasters("SD", "SEVERE STORMS, STRAIGHT-LINE WINDS", "2026-06-15", "2026-07-31")
print(oohtmp)
# HAVE TO USE NOTEBOOK COMPUTE FOR FEMA
this_disaster = int(oohtmp['disasterNumber'][0])
areas = get_disaster_areas(this_disaster)
print(areas)

def padded_fips(series, width):
    return (
        pd.to_numeric(series, errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(width)
    )

# State FIPS = 2 digits; county FIPS = 3 digits
storm_cty = (
    padded_fips(areas["fipsStateCode"], 2)
    + padded_fips(areas["fipsCountyCode"], 3)
)

# Remove missing values and duplicates
storm_cty = storm_cty.dropna().drop_duplicates()

print(storm_cty)

# want to do a query like: 
# https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries?$filter=state%20eq%20%27SD%27%20and%20substringof(%27STORM%27,declarationTitle)%20and%20incidentBeginDate%20ge%202026-06-01%20and%20incidentEndDate%20le%202026-07-31

In [0]:
# get just housing loans 
# prob need to make sure that these loans have an unpaid principal - cuz paid off not really a risk
# oh crud can't get unpaid principal from obligations dataset
# need to get all of the SD loans - and then we need to get the bounds of that storm and filter further
q6 = pfdo.filter(pfdo.portfolio_group == 'Residential')\
         .filter(pfdo.obligation_state_abr == 'SD')\
         .filter(F.col("investment_type").like("%Loan%"))\
         .groupBy('state_name','obligation_county_name', 'obligation_county_fips')\
         .agg(F.sum('obl_amt').alias('historical_housing_loan_investment'))
q6_tot = q6.select(F.sum('historical_housing_loan_investment')).collect()[0][0]
display(q6)
q6 = q6.withColumn('in_bounds', F.col('obligation_county_fips').isin(list(storm_cty)))
q6_storm_tot = q6.filter(q6.in_bounds)\
    .select(F.sum('historical_housing_loan_investment')).collect()[0][0]

print(f'historical housing loans in state: {q6_tot}')
print(f'historical housing loans in storm counties: {q6_storm_tot}')
print(f'percent of state housing loans in storm counties: {q6_storm_tot/q6_tot}')